# Banquet — query-audio inference

Uses the same `wav/mix.wav` and `wav/guide.wav` inputs as the GuideSep notebook, but separates with the Banquet query-based music source separation model. Outputs are written to `output/target_banquet.wav` and `output/residual_banquet.wav`.


In [ ]:
from pathlib import Path
import os
import shutil
import sys
import urllib.request
import zipfile

BANQUET_COMMIT = "79ed5bb75e5c3a40cd319d9d990cee913fc65c26"
RUNTIME_ROOT = Path(".runtime/banquet")
SOURCE_DIR = RUNTIME_ROOT / "query-bandit"

if not (SOURCE_DIR / "train.py").exists():
    RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = RUNTIME_ROOT / "query-bandit.zip"
    archive_url = f"https://github.com/kwatcharasupat/query-bandit/archive/{BANQUET_COMMIT}.zip"

    print("Downloading Banquet source...")
    urllib.request.urlretrieve(archive_url, archive_path)

    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(RUNTIME_ROOT)

    extracted_dir = RUNTIME_ROOT / f"query-bandit-{BANQUET_COMMIT}"
    if SOURCE_DIR.exists():
        shutil.rmtree(SOURCE_DIR)
    extracted_dir.rename(SOURCE_DIR)
    archive_path.unlink(missing_ok=True)

os.environ["CONFIG_ROOT"] = str((SOURCE_DIR / "config").resolve())
os.environ.setdefault("DATA_ROOT", str((RUNTIME_ROOT / "data").resolve()))

source_path = str(SOURCE_DIR.resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)

print("Banquet runtime source:", SOURCE_DIR.resolve())


In [ ]:
import numpy as np
import soundfile as sf
import torch

try:
    import torchaudio as ta
except ImportError as exc:
    raise RuntimeError(
        "Banquet requires torchaudio for its internal STFT and PaSST resampling. "
        "Install the torchaudio build matching your PyTorch/CUDA build, then restart the kernel."
    ) from exc

from IPython.display import Audio, display

# Banquet internally needs torchaudio transforms, but its official inference helper
# also uses torchaudio.load/save. Patch only I/O so TorchCodec is not required.
def _soundfile_load(uri, frame_offset=0, num_frames=-1, normalize=True,
                    channels_first=True, format=None, buffer_size=4096, backend=None):
    start = int(frame_offset)
    frames = -1 if num_frames is None or int(num_frames) < 0 else int(num_frames)
    audio, sr = sf.read(
        str(uri),
        start=start,
        frames=frames,
        always_2d=True,
        dtype="float32",
    )
    tensor = torch.from_numpy(np.ascontiguousarray(audio.T))
    if not channels_first:
        tensor = tensor.T
    return tensor, sr


def _soundfile_save(uri, src, sample_rate, channels_first=True, format=None,
                    encoding=None, bits_per_sample=None, buffer_size=4096,
                    backend=None, compression=None):
    tensor = src.detach().cpu()
    if tensor.ndim == 1:
        audio = tensor.numpy()
    elif tensor.ndim == 2:
        audio = tensor.numpy().T if channels_first else tensor.numpy()
    else:
        raise ValueError(f"Expected 1-D or 2-D audio tensor, got shape {tuple(tensor.shape)}")
    sf.write(str(uri), audio, int(sample_rate), subtype="FLOAT")


ta.load = _soundfile_load
ta.save = _soundfile_save

from train import inference_byoq
from core.models.e2e.bandit.bandit import PasstFiLMConditionedBandit


# Upstream chunked inference passes the same fixed query to every mixture chunk,
# but PasstFiLMConditionedBandit.adapt_query() recomputes the PaSST embedding each time.
# Cache only that query embedding. The separator, chunking, overlap, FiLM conditioning,
# and all model parameters remain unchanged.
def _cached_adapt_query(self, q, batch):
    query_audio = batch.query.audio
    cache_key = (
        query_audio.data_ptr(),
        tuple(query_audio.shape),
        query_audio.device,
        query_audio.dtype,
    )
    cached = getattr(self, "_banquet_query_embedding_cache", None)

    if cached is None or cached[0] != cache_key:
        with torch.inference_mode():
            w = self.query_encoder(query_audio)
        self._banquet_query_embedding_cache = (cache_key, w)
    else:
        w = cached[1]

    q = torch.permute(q, (0, 3, 1, 2))
    q = self.film(q, w)
    q = torch.permute(q, (0, 2, 3, 1))
    return q


PasstFiLMConditionedBandit.adapt_query = _cached_adapt_query

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Banquet optimization: PaSST guide embedding is cached once per model/query.")


In [ ]:
MIX_PATH = Path("wav/mix.wav")
GUIDE_PATH = Path("wav/guide.wav")
TARGET_PATH = Path("output/target_banquet.wav")
RESIDUAL_PATH = Path("output/residual_banquet.wav")

CHECKPOINT_DIR = RUNTIME_ROOT / "checkpoints"
CHECKPOINT_PATH = CHECKPOINT_DIR / "ev-pre-aug.ckpt"
CHECKPOINT_URL = "https://zenodo.org/records/13694558/files/ev-pre-aug.ckpt?download=1"

BATCH_SIZE = 8

if not MIX_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {MIX_PATH.resolve()}")
if not GUIDE_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {GUIDE_PATH.resolve()}")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if not CHECKPOINT_PATH.exists():
    print("Downloading Banquet ev-pre-aug checkpoint (~646 MB)...")
    urllib.request.urlretrieve(CHECKPOINT_URL, CHECKPOINT_PATH)

print("Checkpoint:", CHECKPOINT_PATH.resolve())
print("Mixture:", sf.info(str(MIX_PATH)))
print("Guide:", sf.info(str(GUIDE_PATH)))


In [ ]:
# The published Banquet model is stereo (in_channel=2). If the mixture is mono,
# duplicate it to stereo for inference; the final outputs are converted back to mono.
original_mix, original_sr = sf.read(str(MIX_PATH), always_2d=True, dtype="float32")
original_channels = original_mix.shape[1]

if original_channels not in (1, 2):
    raise ValueError(
        f"Banquet expects mono or stereo input here; got {original_channels} channels."
    )

runtime_mix_path = MIX_PATH
if original_channels == 1:
    runtime_mix_path = RUNTIME_ROOT / "mono_as_stereo_mix.wav"
    stereo_mix = np.repeat(original_mix, 2, axis=1)
    sf.write(str(runtime_mix_path), stereo_mix, original_sr, subtype="FLOAT")

TARGET_PATH.parent.mkdir(parents=True, exist_ok=True)

inference_byoq(
    ckpt_path=str(CHECKPOINT_PATH.resolve()),
    input_path=str(runtime_mix_path.resolve()),
    query_path=str(GUIDE_PATH.resolve()),
    output_path=str(TARGET_PATH.resolve()),
    config_path=str((SOURCE_DIR / "expt" / "bandit-everything-test.yml").resolve()),
    stem_name="target",
    batch_size=BATCH_SIZE,
    use_cuda=torch.cuda.is_available(),
)

print("Banquet inference complete.")


In [ ]:
target, target_sr = sf.read(str(TARGET_PATH), always_2d=True, dtype="float32")
if target_sr != original_sr:
    raise RuntimeError(
        f"Unexpected target sample rate {target_sr}; expected original mixture rate {original_sr}."
    )

# Restore the original channel layout.
if original_channels == 1:
    target = target.mean(axis=1, keepdims=True)
elif target.shape[1] == 1:
    target = np.repeat(target, 2, axis=1)
elif target.shape[1] > 2:
    target = target[:, :2]

# Match the original mixture length exactly.
n_samples = original_mix.shape[0]
if target.shape[0] < n_samples:
    target = np.pad(target, ((0, n_samples - target.shape[0]), (0, 0)))
else:
    target = target[:n_samples]

residual = original_mix - target

sf.write(str(TARGET_PATH), target, original_sr, subtype="FLOAT")
sf.write(str(RESIDUAL_PATH), residual, original_sr, subtype="FLOAT")

print("Saved:", TARGET_PATH.resolve())
print("Saved:", RESIDUAL_PATH.resolve())
print(f"Duration: {n_samples / original_sr:.3f} s")

print("Target")
display(Audio(target.T if target.shape[1] > 1 else target[:, 0], rate=original_sr))
print("Residual")
display(Audio(residual.T if residual.shape[1] > 1 else residual[:, 0], rate=original_sr))
